# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}\n")
print(f"DOI/Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

The `mlcroissant` library exposes available record sets via the metadata. For each record set, we can list its `@id`, name, and its available fields (their `@id` and name).

In [ ]:
# List all record sets and their fields by `@id`

record_sets_info = []
record_sets = getattr(metadata, 'record_sets', [])

if not record_sets:
    # Fallback for croissant 1.0 metadata
    record_sets = getattr(metadata, 'recordSet', [])

print("Available record sets in the dataset:")
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', rs_id)
    print(f"- Record set @id: {rs_id}, name: {rs_name}")
    fields = getattr(rs, 'fields', None)
    # Fallback for Croissant 1.0: 'field' or 'fields' may be used
    if fields is None:
        fields = getattr(rs, 'field', [])
    if fields:
        for f in fields:
            f_id = getattr(f, '@id', None)
            f_name = getattr(f, 'name', None)
            print(f"    - field @id: {f_id}, name: {f_name}")
    else:
        print("    - No fields listed.")
    record_sets_info.append({'@id': rs_id, 'name': rs_name})

# For further steps, collect record set @ids
record_set_ids = [r['@id'] for r in record_sets_info if r['@id']]

In [ ]:
# Show a sample of the records for each available record set
# Use the first available record set as example

for rs_id in record_set_ids:
    print(f"\nSample records from record set '@id': {rs_id}")
    # Use .records() generator
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            pprint.pprint(rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.

We will extract all record sets available in the metadata, referenced strictly by their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
print("\nExtracting record sets into pandas DataFrames:")

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"- '@id': {rs_id} DataFrame shape: {df.shape}")
        else:
            print(f"- '@id': {rs_id}: No records found.")
    except Exception as e:
        print(f"- '@id': {rs_id}: Error loading records ({e})")

# Show columns for the first non-empty DataFrame
for rs_id, df in dataframes.items():
    print(f"\nColumns in record set '@id': {rs_id}\n{df.columns.tolist()}")
    display(df.head(3))
    break # Just display for one for demonstration

## 4. Exploratory Data Analysis (EDA)
In this section, we'll demonstrate how to process a numeric field. We'll select a record set and one of its fields (referenced by `@id`), apply filtering, normalization, and groupby operations.

You can adapt code and field selection based on the actual fields present in the dataset.

In [ ]:
# Example EDA: select a numeric field (by @id) and group/categorize the data

# Pick the first DataFrame loaded
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id} (shape: {df.shape})")

    # Show all available columns (field @ids):
    print("\nAvailable fields in this record set:")
    for i, col in enumerate(df.columns):
        print(f"  {i+1}. {col}")
    # Try to select a numeric field by inferring type from pandas dtype
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"\nSelected numeric field (by @id): {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
        display(filtered_df[[numeric_field]].head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to use any categorical (object) field for grouping
        group_candidates = [col for col in df.columns if df[col].dtype == 'O']
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping filtered data by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            display(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
    else:
        print("No numeric field found for EDA demonstration.")
else:
    print("No dataframes loaded for analysis.")

## 5. Visualization
Visualize data distributions or relationships. For demonstration, we will plot the distribution of a numeric field in one of the record sets.

You may adjust the code below to use a specific `@id` as required.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If a numeric field was found earlier, plot its distribution
if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of '{numeric_field}' in record set '@id': {record_set_id}")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load a Croissant-compliant dataset with mlcroissant using its schema URL.
- Inspect the dataset metadata, record sets, and field `@id`s.
- Extract all record sets to pandas DataFrames for further analysis.
- Conduct EDA, including filtering and normalization, referencing fields by their `@id`.
- Visualize the distribution of numeric variables.

For in-depth research or production use, continue exploring other record sets or fields referenced by their `@id`, and adapt your analysis code as needed.